# 01 · Neural Relation Operators — Toy Experiment

**Hypothesis**: A new composite relation can be learned from very few examples  
by initialising its operator as a composition of existing operators.

This notebook is a clean wrapper around `mre.knowledge_graph`.  
All model/training code lives in the package — the notebook only orchestrates.

## Experiment design

| Symbol | Relation | Meaning |
|--------|----------|---------|
| D | `depends_on` | theorem A's proof requires theorem B |
| G | `generalizes` | theorem A subsumes theorem B |
| E | `equivalent_to` | A and B are equivalent |
| P | `applied_in` | technique A is used in proof of B |

Composite (never seen during base training):
```
gen_dep = generalizes ∘ depends_on
```

Three strategies compared across k = {1, 5, 10, 25, 50, 100, full}:
1. **from_scratch** — random init, fine-tune on k examples  
2. **compose_fixed** — `f_gen ∘ f_dep`, no fine-tuning  
3. **compose_finetune** — compositional init → fine-tune on k examples  

In [ ]:
# ── Environment setup ──────────────────────────────────────────────────────
import sys, os
# Make the package importable when running inside Kaggle or a notebook
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import copy
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from mre.utils import load_config, resolve_device, set_seed, get_logger
from mre.knowledge_graph import (
    NROModel, SyntheticKG,
    hits_at_k, train_epoch, train_base_model,
)

# ── Config & reproducibility ───────────────────────────────────────────────
cfg    = load_config()                   # reads configs/kaggle_config.yaml
DEVICE = resolve_device(cfg)
set_seed(cfg.experiment.seed)
logger = get_logger('notebook_01')

print(f'Device : {DEVICE}')
print(f'Config : env={cfg.env}')

## 1 · Synthetic Knowledge Graph

In [ ]:
kg = SyntheticKG(
    num_entities = cfg.knowledge_graph.max_entities,
    embed_dim    = cfg.knowledge_graph.embed_dim,
    noise_std    = cfg.knowledge_graph.noise_std,
    seed         = cfg.experiment.seed,
)
print(kg.summary())

## 2 · Phase 1 — Train Base Model on 4 Relations

In [ ]:
BASE_RELATIONS = cfg.nro.base_relations

# Prepare splits
splits = {}
for rel in BASE_RELATIONS:
    train, val, test = kg.get_split(rel)
    splits[rel] = {'train': train, 'val': val, 'test': test}
    print(f'  {rel:20s}  train={len(train)}  val={len(val)}  test={len(test)}')

model = NROModel(
    num_entities   = kg.num_entities,
    relation_names = BASE_RELATIONS,
    embed_dim      = cfg.nro.embed_dim,
    hidden_dim     = cfg.nro.hidden_dim,
).to(DEVICE)

print(f'\nModel parameters: {sum(p.numel() for p in model.parameters()):,}')

history = train_base_model(
    model,
    splits,
    BASE_RELATIONS,
    epochs     = cfg.nro.train_epochs,
    lr         = cfg.nro.learning_rate,
    weight_decay = cfg.nro.weight_decay,
    batch_size = cfg.nro.batch_size,
    hits_k     = cfg.experiment.hits_k,
)

print('\nFinal test Hits@10 (base relations):')
for rel in BASE_RELATIONS:
    h = hits_at_k(model, splits[rel]['test'], [rel], k=cfg.experiment.hits_k)
    print(f'  {rel:20s}: {h:.3f}')

## 3 · Phase 2 — Composition Hypothesis Experiment

In [ ]:
from mre.knowledge_graph.experiments import run_composition_experiment

comp_train, comp_val, comp_test = kg.get_split(cfg.experiment.composite_relation)
print(f'{cfg.experiment.composite_relation} — train:{len(comp_train)}  val:{len(comp_val)}  test:{len(comp_test)}')

results = run_composition_experiment(
    base_model       = model,
    comp_relation    = cfg.experiment.composite_relation,
    comp_chain       = cfg.experiment.composite_chain,
    train_pool       = comp_train,
    val_edges        = comp_val,
    test_edges       = comp_test,
    k_shots          = cfg.experiment.k_shots,
    n_trials         = cfg.experiment.n_trials,
    ft_epochs        = cfg.nro.finetune_epochs,
    ft_lr            = cfg.nro.finetune_lr,
    hits_k           = cfg.experiment.hits_k,
)

## 4 · Results Summary

In [ ]:
from mre.knowledge_graph.experiments import print_results_table, plot_results

print_results_table(results, cfg.experiment.k_shots)
fig = plot_results(results, history, cfg.experiment.k_shots)
fig.savefig('NRO_results.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Figure saved to NRO_results.png')

## 5 · Ablation — `indirect_dep = depends_on ∘ depends_on`

In [ ]:
abl_train, abl_val, abl_test = kg.get_split('indirect_dep')

abl_results = run_composition_experiment(
    base_model    = model,
    comp_relation = 'indirect_dep',
    comp_chain    = ['depends_on', 'depends_on'],
    train_pool    = abl_train,
    val_edges     = abl_val,
    test_edges    = abl_test,
    k_shots       = [1, 5, 10, 50, 'full'],
    n_trials      = cfg.experiment.n_trials,
    ft_epochs     = cfg.nro.finetune_epochs,
    ft_lr         = cfg.nro.finetune_lr,
    hits_k        = cfg.experiment.hits_k,
)

print('\nAblation — indirect_dep:')
print_results_table(abl_results, [1, 5, 10, 50, 'full'])